In [1]:
!pip install -q torch torchvision tqdm


In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
from tqdm import tqdm
import os


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


device(type='cuda')

In [4]:
DATA_DIR = "classification_data"
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 15
NUM_CLASSES = 25


In [5]:
train_transforms = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


In [8]:
!unzip class_sample.zip

Streaming output truncated to the last 5000 lines.
  inflating: classification_data/train/dog/dog_140.jpg  
  inflating: classification_data/train/dog/dog_94.jpg  
  inflating: classification_data/train/dog/dog_304.jpg  
  inflating: classification_data/train/dog/dog_191.jpg  
  inflating: classification_data/train/dog/dog_125.jpg  
  inflating: classification_data/train/dog/dog_315.jpg  
  inflating: classification_data/train/dog/dog_349.jpg  
  inflating: classification_data/train/dog/dog_88.jpg  
  inflating: classification_data/train/dog/dog_184.jpg  
  inflating: classification_data/train/dog/dog_194.jpg  
  inflating: classification_data/train/dog/dog_216.jpg  
  inflating: classification_data/train/dog/dog_235.jpg  
  inflating: classification_data/train/dog/dog_84.jpg  
  inflating: classification_data/train/dog/dog_203.jpg  
  inflating: classification_data/train/dog/dog_106.jpg  
  inflating: classification_data/train/dog/dog_312.jpg  
  inflating: classification_data/train/d

In [9]:
train_dataset = datasets.ImageFolder(
    os.path.join(DATA_DIR, "train"),
    transform=train_transforms
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

print("Train samples:", len(train_dataset))
print("Classes:", train_dataset.classes)


Train samples: 8750
Classes: ['airplane', 'bed', 'bench', 'bicycle', 'bird', 'bottle', 'bowl', 'bus', 'cake', 'car', 'cat', 'chair', 'couch', 'cow', 'cup', 'dog', 'elephant', 'horse', 'motorcycle', 'person', 'pizza', 'potted plant', 'stop sign', 'traffic light', 'truck']


In [10]:
model = models.mobilenet_v2(pretrained=True)

# Freeze entire base
for param in model.features.parameters():
    param.requires_grad = False

model


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V2_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V2_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 66.3MB/s]


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [11]:
model.classifier[1] = nn.Linear(
    model.classifier[1].in_features,
    NUM_CLASSES
)

model = model.to(device)
model.classifier


Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=25, bias=True)
)

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-4)


In [16]:
best_train_acc = 0.0

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    avg_loss = running_loss / len(train_loader)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Loss: {avg_loss:.4f} | Train Acc: {train_acc:.4f}")

    if train_acc > best_train_acc:
        best_train_acc = train_acc
        torch.save(model.state_dict(), "mobilenetv2_best.pth")


Epoch 1/15: 100%|██████████| 274/274 [00:24<00:00, 11.06it/s]


Epoch [1/15] | Loss: 2.9878 | Train Acc: 0.1697


Epoch 2/15: 100%|██████████| 274/274 [00:23<00:00, 11.89it/s]


Epoch [2/15] | Loss: 2.5537 | Train Acc: 0.3375


Epoch 3/15: 100%|██████████| 274/274 [00:23<00:00, 11.90it/s]


...

Epoch [13/15] | Loss: 1.7247 | Train Acc: 0.5007


Epoch 14/15: 100%|██████████| 274/274 [00:23<00:00, 11.78it/s]


Epoch [14/15] | Loss: 1.7022 | Train Acc: 0.5051


Epoch 15/15: 100%|██████████| 274/274 [00:23<00:00, 11.75it/s]

Epoch [15/15] | Loss: 1.6853 | Train Acc: 0.5083


In [18]:
from google.colab import files
files.download("mobilenetv2_best.pth")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>